# Design and implementation assumptions

## Retained scope and source interpretation

This attempt covers the four frozen Table 2 regressions: Panel A bad news and Panel B good news, each with actual and simulated returns, using the replicated 1963–1990 sample. Table 2 columns (2) and (3) report the retained fits (physical PDF p. 20). Later-period columns and Table 3 are outside this attempt.

The selected dependent variable is `X = item 18 earnings / firm market value at FYE−9`. The Table 2 note on physical p. 20 explicitly identifies net income before extraordinary items and discontinued operations (Compustat #18), scaled by firm market value nine months before fiscal year-end. Table 1’s note on physical p. 12 instead says stock price. I follow the more specific Table 2 definition for the selected regressions and document the conflict here. The provider field crosswalk maps current `COMP.FUNDA.IB` to annual item A18. Whether this current field's vintage, restatement history, and definition are equivalent to the authors' historical item 18 data remains uncertain.

Table 1, physical p. 12, says observations in the top or bottom 1% of each variable within each year are excluded. I apply within-fiscal-year 1st and 99th percentile cutoffs to `X` and actual `R`, excluding values outside the cutoffs before forming the actual-return partitions. This is a trim, not winsorization. The full trimmed actual sample supports one pooled generator regression, `R = b0 + b1 X + residual`, as described in Table 2 Panel C, physical p. 20. Table 2, physical pp. 19–20, says simulated returns add a same-year residual from another firm without replacement. I use one deterministic, seeded, within-year residual derangement and do not trim simulated returns a second time. Simulated observations are partitioned by simulated-return sign.

## Data, dates, units, and joins

The construction reads `COMP.FUNDA`, `COMP.G_EXRT_MTH`, `CRSP.MSF`, and `CRSP.CCMXPF_LINKTABLE` via exact catalog paths and read-only SAS libraries. A separate bounded audit of the final company-equity constituents also reads catalog-listed `CRSP.MSENAMES` for date-effective share codes and security identifiers. I select annual industrial, consolidated, domestic, standard-format Compustat rows (`INDL`, `C`, `D`, `STD`) for fiscal years 1963–1990. I require populated `IB` and own-period `CURCD` before resolving repeated company-year rows; among complete rows, I retain the latest fiscal end date in the current data vintage. This prevents an empty companion record from displacing a populated statement and keeps earnings and currency from the same source row. The file’s own-period `CURCD` controls conversion; `CURNCD` is not substituted. USD is an identity conversion. For other reporting currencies, I apply the provider’s annual income and cash-flow rule using `EXRAT12M` legs at the statement’s reporting month, with GBP as the intermediate currency: `IB_USD = IB / (GBP→CURCD) × (GBP→USD)`. GBP reporting currency uses only the GBP-to-USD leg. I do not require twelve separate monthly quotes. The local provider guide names `EXRAT12M` for annual income and cash-flow translation and supplies both legs in the same monthly exchange-rate table (PDF pp. 87–88; printed pp. 79–80). `IB` is treated as millions of reporting currency and converted to USD thousands. A diagnostic-only calculation compares this construction with A1’s 12-month average of `EXRAT1M` cross-rate ratios; it does not feed the analysis data.

The earnings denominator follows Table 2: company market equity at the FYE−9 month. `CRSP.MSF.PRC` is used in absolute value because a negative CRSP price denotes a bid/ask average; `SHROUT` is in thousands of shares, so `abs(PRC) × SHROUT` is USD thousands. A date-valid CCM primary route identifies the CRSP `PERMCO`. Company market equity sums each distinct CRSP monthly issue sharing that `PERMCO` in the anchor month, requiring positive price and shares for every included issue. This allows secondary share classes with no separate CCM primary route into the company total. `MSF` itself has no `SHRCD`, but catalog-listed `MSENAMES` supplies `SHRCD`, `NAMEDT`, `NAMEENDT`, and `PERMNO`; its effective row can be matched to each constituent at the actual `MSF.DATE`. The date-qualified final-sample audit finds certificates (codes 20/21) and ADRs (30/31) in 437 issue-months across 431 company-months. Four company-months also have a 10/11/12 ordinary-common or code 18 REIT issue: code 21 coexists with code 11 in three, and code 31 with code 12 in one. Those four issue pairs have no matching `NCUSIP`, CUSIP header, MSF CUSIP, or `SHRCLS`. The remaining 427 certificate/ADR company-months have no 10/11/12/18 issue recorded. The source does not specify a share-code screen or ADR ratio, so PERMCO co-occurrence alone does not establish duplicate valuation. All distinct issues remain included, and the source-silent treatment of certificate, ADR, beneficial-interest, unit, and fund securities remains a company-MVE coverage limitation. The CRSP Stock and Index Data Description Guide (physical PDF p. 115; printed p. 111) classifies codes 14/15 and 74 as closed-end-fund/UIT subtypes; 80 such issue-months occur in the final inputs, including two code 74 issue-months. The paper does not explicitly exclude funds. The build reports company versus primary-security market equity differences and keeps both measures for audit.

For actual returns, the selected return security is the unique top-ranked link effective at the FYE−9 anchor: `LINKPRIM=P` before `C`, and `LINKTYPE=LC` before `LU`. Tied best-ranked security/company links are treated as ambiguous. The selected security must map to the same PERMCO at the anchor and have a CCM link effective on each CRSP monthly observation date in the required window. I preserve open CCM end dates coded `.E`. The 12 monthly return periods are FYE−8 through FYE+3; the anchor price/equity month is FYE−9. Each actual return compounds 12 distinct, nonmissing CRSP `RET` observations. The fiscal-year label sets the 1963–1990 sample period; actual FYE dates and the needed CRSP history can extend through 1991.

## Estimation and outputs

Each retained regression is OLS with an intercept and ordinary Stata inference. The paper reports regressions and t-statistics but does not name an estimator, covariance correction, or p-value sidedness. I do not add fixed effects, weights, or clustering. The two panel tables report the retained slope coefficients with p-values, N, R-squared, and adjusted R-squared in the template’s four formats. Native SAS output records the pooled supporting generator fit and construction counts. Descriptive outputs cover the full actual sample and the four actual/simulated sign partitions, including `X`, actual `R`, and simulated `R`.

## Assumptions and limitations

- The provider field crosswalk maps current `IB` to A18. Historical-vintage, restatement, and definition equivalence to the authors' item 18 data remains uncertain.
- The Table 2 MVE wording controls over Table 1’s price wording. All distinct issues on the date-valid CCM PERMCO are retained because the paper does not specify an issue-type screen and no double-counted claim was demonstrated in the date-qualified final-sample audit. The MVE sum should still be interpreted with the source-silent share-class limitation above.
- The link ranking is an explicit best-effort rule where the paper does not specify link selection. Tied top-ranked links are omitted rather than chosen arbitrarily.
- The paper does not document its treatment of non-USD reporting currency; the provider's documented period-rate method is used. Current-vintage `EXRAT12M` coverage and measured rates may differ from the authors' data.
- The paper does not report a simulation seed. Seed `20261103` is fixed for reproducibility, selected independently of published sample counts or estimates. A random ordering followed by a one-step cyclic rotation assigns every same-year residual once and avoids the recipient firm; years with one eligible firm cannot produce a valid assignment.
- Percentiles use SAS `PROC MEANS` P1/P99 defaults. Differences in historical data vintage, date-valid security links, available currency rates, and the source-silent ordinary inference convention may produce differences from the published table.
- Only the single baseline simulation is run; across-year residual assignment, 1,000 repetitions, and the fixed-slope robustness are not part of this retained scope.


In [ ]:
options errorabend;

/* Read only the catalog-listed input tables needed for the 1963-1990 baseline. */
libname comp "/storage/sas/base/comp/d_na" access=readonly;
libname crsp "/storage/sas/base/crsp/a_stock" access=readonly;
libname ccm "/storage/sas/base/crsp/a_ccm" access=readonly;
libname fxsrc "/storage/sas/base/comp/d_global/currency" access=readonly;
libname out "data";

/* One annual industrial-format observation per Compustat company-year.
   If a fiscal-year label has multiple end dates in the current vintage, retain
   its latest annual period end; ties at the same date use the latest FDATE. */
data funda_candidate;
  set comp.funda(keep=gvkey datadate fyear fdate indfmt consol popsrc datafmt
                      curcd pddur ib);
  where fyear>=1963 and fyear<=1990
    and indfmt='INDL' and consol='C' and popsrc='D' and datafmt='STD';
  if not missing(gvkey) and not missing(datadate);
  curcd=upcase(strip(curcd));
  format datadate date9.;
run;
proc sql;
  title 'Compustat candidate field completeness before annual-row selection';
  select count(*) as candidate_rows,
         sum(missing(ib)) as missing_ib,
         sum(missing(curcd)) as missing_curcd
  from funda_candidate;
  title 'GVKEY 006557 duplicate-year completeness audit';
  select fyear,count(*) as candidate_rows,
         sum(not missing(ib)) as populated_ib,
         sum(not missing(curcd)) as populated_curcd,
         sum(not missing(ib) and not missing(curcd)) as complete_model_rows,
         sum(curcd='CAD') as cad_rows,sum(curcd='USD') as usd_rows,
         min(datadate) as earliest_date format=date9.,
         max(datadate) as latest_date format=date9.
  from funda_candidate
  where gvkey='006557'
  group by fyear order by fyear;
quit;
data funda_raw;
  set funda_candidate;
  if not missing(ib) and not missing(curcd);
run;
proc sort data=funda_raw out=funda_sort;
  by gvkey fyear datadate fdate;
run;
data funda_latest_date;
  set funda_sort;
  by gvkey fyear datadate;
  if last.datadate;
run;
proc sql;
  create table funda_year_counts as
  select gvkey,fyear,count(*) as n_dates
  from funda_latest_date
  group by gvkey,fyear;
  title 'Repeated complete annual rows before fiscal-year selection';
  select count(*) as company_year_groups,
         sum(n_dates>1) as repeated_company_year_groups,
         sum(n_dates) as complete_annual_rows
  from funda_year_counts;
quit;
proc sort data=funda_latest_date out=funda_yearsort;
  by gvkey fyear datadate;
run;
data funda_annual;
  set funda_yearsort;
  by gvkey fyear;
  if last.fyear;
  obsid=_n_;
  anchor_mth=intnx('month',datadate,-9,'b');
  ret_first=intnx('month',datadate,-8,'b');
  ret_last=intnx('month',datadate,3,'b');
  fx_first=intnx('month',datadate,-11,'b');
  fx_last=intnx('month',datadate,0,'b');
  format datadate anchor_mth ret_first ret_last fx_first fx_last date9.;
run;

/* Translate annual income with provider EXRAT12M legs at the statement's own
   reporting month. GBP is the intermediate currency. USD is an identity
   conversion; CURNCD is not substituted for the record's own CURCD. */
data fx_raw;
  set fxsrc.g_exrt_mth(keep=datadate fromcurm tocurm exrat12m);
  if upcase(strip(fromcurm))='GBP' and not missing(tocurm)
     and not missing(exrat12m) and exrat12m>0;
  report_month=intnx('month',datadate,0,'b');
  curcd=upcase(strip(tocurm));
  rate=exrat12m;
  keep datadate report_month curcd rate;
  format datadate report_month date9.;
run;
proc sql;
  create table fx_agg as
  select report_month,curcd,min(datadate) as first_quote,
         max(datadate) as last_quote,min(rate) as rate,
         max(rate) as rate_max,count(*) as nquote
  from fx_raw
  group by report_month,curcd;
  create table fx_valid as
  select report_month,curcd,first_quote,last_quote,rate,nquote
  from fx_agg
  where rate=rate_max;
  create table fx_factor as
  select a.report_month,a.curcd,a.rate as gbp_to_reporting,
         b.rate as gbp_to_usd,b.rate/a.rate as usd_factor,
         a.nquote as quote_reporting,b.nquote as quote_usd
  from fx_valid as a inner join fx_valid as b
    on a.report_month=b.report_month
  where b.curcd='USD' and a.curcd not in ('USD','GBP')
    and a.rate>0 and b.rate>0
  union all
  select report_month,'GBP' as curcd,1 as gbp_to_reporting,
         rate as gbp_to_usd,rate as usd_factor,
         0 as quote_reporting,nquote as quote_usd
  from fx_valid where curcd='USD' and rate>0;
  create table fx_for_year as
  select a.obsid,count(b.report_month) as fx_obs,
         min(b.report_month) as fx_report_month,
         min(b.gbp_to_reporting) as fx_gbp_to_reporting,
         min(b.gbp_to_usd) as fx_gbp_to_usd,
         min(b.usd_factor) as usd_factor,
         min(b.quote_reporting) as fx_quote_reporting,
         min(b.quote_usd) as fx_quote_usd
  from funda_annual as a left join fx_factor as b
    on a.curcd=b.curcd
   and b.report_month=intnx('month',a.datadate,0,'b')
  where a.curcd ne 'USD'
  group by a.obsid;
quit;
proc sort data=funda_annual; by obsid; run;
proc sort data=fx_for_year; by obsid; run;
data funda_usd;
  merge funda_annual(in=a) fx_for_year;
  by obsid;
  if a;
  if curcd='USD' then do;
    usd_factor=1;
    fx_obs=1;
    fx_report_month=intnx('month',datadate,0,'b');
    fx_quote_reporting=0;
    fx_quote_usd=0;
  end;
  if fx_obs=1 and usd_factor>0 then ib_usd_th=ib*1000*usd_factor;
  else ib_usd_th=.;
  format fx_report_month date9.;
  label ib_usd_th='Item 18 proxy in USD thousands';
run;
proc sql;
  title 'EXRAT12M quote uniqueness and leg coverage';
  select count(*) as report_month_currency_pairs,
         sum(nquote>1) as pairs_with_duplicate_rows,
         sum(rate ne rate_max) as pairs_with_conflicting_rates,
         min(nquote) as minimum_quote_rows,max(nquote) as maximum_quote_rows
  from fx_agg;
  title 'Provider period-rate conversion coverage by own-period CURCD';
  select curcd,count(*) as annual_rows,
         sum(fx_obs=1 and not missing(ib_usd_th)) as converted_rows,
         sum(fx_obs=0) as no_period_rate_rows
  from funda_usd group by curcd order by curcd;
  title 'GVKEY 006557 selected statement-month EXRAT12M legs';
  select gvkey,fyear,datadate,curcd,ib,fx_obs,fx_report_month,
         fx_gbp_to_reporting,fx_gbp_to_usd,usd_factor,ib_usd_th,
         fx_quote_reporting,fx_quote_usd
  from funda_usd
  where gvkey='006557' and fyear in (1982,1983)
  order by fyear;
quit;

/* Audit-only reconstruction of A1's 12 monthly EXRAT1M cross-rate averages.
   These values do not feed the handoff. They quantify coverage and rate
   differences against the documented statement-month EXRAT12M construction. */
data fx1m_raw;
  set fxsrc.g_exrt_mth(keep=datadate fromcurm tocurm exrat1m
      where=(datadate>='01jan1962'd and datadate<='31dec1991'd));
  if upcase(strip(fromcurm))='GBP' and not missing(tocurm)
     and not missing(exrat1m) and exrat1m>0;
  month=intnx('month',datadate,0,'b');
  curcd=upcase(strip(tocurm));
  rate=exrat1m;
  keep month curcd rate;
  format month date9.;
run;
proc sql;
  create table fx1m_agg as
  select month,curcd,min(rate) as rate,max(rate) as rate_max,count(*) as nquote
  from fx1m_raw group by month,curcd;
  create table fx1m_valid as
  select month,curcd,rate
  from fx1m_agg where rate=rate_max;
  create table fx1m_factor as
  select a.month,a.curcd,b.rate/a.rate as usd_factor
  from fx1m_valid as a inner join fx1m_valid as b
    on a.month=b.month
  where b.curcd='USD' and a.curcd not in ('USD','GBP') and a.rate>0
  union all
  select month,'GBP' as curcd,rate as usd_factor
  from fx1m_valid where curcd='USD';
  create table fx1m_for_year as
  select a.obsid,avg(b.usd_factor) as old_avg_usd_factor,
         count(distinct b.month) as old_fx_months
  from funda_annual as a left join fx1m_factor as b
    on a.curcd=b.curcd and b.month between a.fx_first and a.fx_last
  where a.curcd ne 'USD'
  group by a.obsid;
  create table fx_comparison as
  select a.obsid,a.gvkey,a.fyear,a.datadate,a.curcd,a.ib,a.fx_obs,
         a.usd_factor,a.ib_usd_th,
         case when a.curcd='USD' then 1 else b.old_avg_usd_factor end
           as old_avg_usd_factor,
         case when a.curcd='USD' then 12 else coalesce(b.old_fx_months,0) end
           as old_fx_months
  from funda_usd as a left join fx1m_for_year as b on a.obsid=b.obsid;
  title 'Provider-period versus A1 monthly-average conversion impact';
  select curcd,count(*) as annual_rows,
         sum(fx_obs=1) as provider_converted,
         sum(old_fx_months=12 and old_avg_usd_factor>0)
           as a1_twelve_month_converted,
         sum(fx_obs=1 and old_fx_months<12)
           as provider_only_after_monthly_gate,
         sum(fx_obs=0 and old_fx_months=12 and old_avg_usd_factor>0)
           as a1_only_after_provider_rule,
         sum(fx_obs=1 and old_fx_months=12 and old_avg_usd_factor>0
             and abs(usd_factor-old_avg_usd_factor)>1e-12)
           as different_complete_rate_factors
  from fx_comparison group by curcd order by curcd;
  title 'GVKEY 006557 old monthly-average audit versus provider period rate';
  select a.gvkey,a.fyear,a.datadate,a.curcd,a.fx_obs,a.fx_report_month,
         a.fx_gbp_to_reporting,a.fx_gbp_to_usd,a.usd_factor,a.ib_usd_th,
         b.old_fx_months,b.old_avg_usd_factor,
         case when b.old_fx_months=12 and b.old_avg_usd_factor>0
              then a.ib*1000*b.old_avg_usd_factor else . end
           as a1_eligible_ib_usd_th
  from funda_usd as a left join fx_comparison as b on a.obsid=b.obsid
  where a.gvkey='006557' and a.fyear in (1982,1983)
  order by a.fyear;
quit;

/* CCM links are retained only for standard CUSIP/unverified links with a
   primary or secondary primary marker. Special open-ended .E is handled at
   each dated CRSP observation; no current/header identifier is used. */
data linkbase;
  set ccm.ccmxpf_linktable(keep=gvkey linkprim linktype lpermno lpermco
                                 linkdt linkenddt);
  if linkprim in ('P','C') and linktype in ('LC','LU')
     and not missing(lpermno) and not missing(lpermco);
run;

/* Exact FYE-9 month anchor: use the CRSP monthly observation in that period.
   CCM supplies the return security and PERMCO. */
proc sql;
  create table anchor_candidates as
  select distinct f.obsid,f.gvkey,f.fyear,f.datadate,f.anchor_mth,
         l.lpermno as permno,l.lpermco as permco,
         l.linkprim,l.linktype,
         case when l.linkprim='P' then 1 else 2 end as prim_rank,
         case when l.linktype='LC' then 1 else 2 end as type_rank,
         m.date as anchor_date,abs(m.prc)*m.shrout as primary_mve
  from funda_usd as f
  inner join linkbase as l on f.gvkey=l.gvkey
  inner join crsp.msf as m on m.permno=l.lpermno and m.permco=l.lpermco
  where intnx('month',m.date,0,'b')=f.anchor_mth
    and (l.linkdt=.B or l.linkdt<=m.date)
    and (l.linkenddt=.E or l.linkenddt>=m.date)
    and m.date between '01jan1962'd and '31dec1991'd;
quit;
proc sql;
  create table anchor_by_issue as
  select obsid,gvkey,fyear,datadate,anchor_mth,permno,permco,
         min(prim_rank) as prim_rank,min(type_rank) as type_rank,
         max(anchor_date) as anchor_date,max(primary_mve) as primary_mve,
         count(distinct anchor_date) as anchor_dates
  from anchor_candidates
  group by obsid,gvkey,fyear,datadate,anchor_mth,permno,permco;
  create table anchor_best_prim as
  select obsid,min(prim_rank) as prim_rank
  from anchor_by_issue group by obsid;
  create table anchor_prim as
  select a.* from anchor_by_issue as a inner join anchor_best_prim as b
    on a.obsid=b.obsid and a.prim_rank=b.prim_rank;
  create table anchor_best_type as
  select obsid,min(type_rank) as type_rank
  from anchor_prim group by obsid;
  create table anchor_top as
  select a.* from anchor_prim as a inner join anchor_best_type as b
    on a.obsid=b.obsid and a.type_rank=b.type_rank;
  create table anchor_rank_count as
  select obsid,count(distinct permno) as n_top_permno,
         count(distinct permco) as n_top_permco
  from anchor_top group by obsid;
  create table ret_choice as
  select a.obsid,a.gvkey,a.fyear,a.datadate,a.anchor_mth,a.permno,a.permco,
         a.prim_rank,a.type_rank,a.anchor_date,a.primary_mve,
         b.n_top_permno,b.n_top_permco
  from anchor_top as a inner join anchor_rank_count as b
    on a.obsid=b.obsid
  where b.n_top_permno=1 and b.n_top_permco=1;
quit;

/* For company MVE, identify the company through the date-valid CCM PERMCO,
   then aggregate distinct CRSP PERMNOs on that PERMCO at the same anchor
   month. This includes eligible secondary issues even when CCM has no separate
   primary route for each class. */
proc sql;
  create table company_anchor_issues as
  select distinct c.obsid,c.gvkey,c.fyear,c.datadate,c.anchor_mth,
         c.permno as primary_permno,c.permco,c.anchor_date,
         m.permno,m.date,abs(m.prc)*m.shrout as issue_mve
  from ret_choice as c
  inner join crsp.msf as m on m.permco=c.permco
  where intnx('month',m.date,0,'b')=c.anchor_mth
    and m.date between '01jan1962'd and '31dec1991'd;
  create table company_anchor_unique as
  select obsid,gvkey,fyear,datadate,anchor_mth,anchor_date,primary_permno,permco,
         permno,max(date) as quote_date,max(issue_mve) as issue_mve
  from company_anchor_issues
  group by obsid,gvkey,fyear,datadate,anchor_mth,anchor_date,primary_permno,permco,permno;
  create table mve_stats as
  select obsid,gvkey,fyear,datadate,anchor_mth,anchor_date,primary_permno,permco,
         count(*) as n_issues,
         sum(case when issue_mve>0 then 1 else 0 end) as n_valid_issues,
         sum(issue_mve) as company_mve_th,
         max(case when permno=primary_permno then issue_mve else . end) as primary_mve_th
  from company_anchor_unique
  group by obsid,gvkey,fyear,datadate,anchor_mth,anchor_date,primary_permno,permco;
quit;

/* Fetch exactly the twelve CRSP months FYE-8 through FYE+3 and require the
   chosen CCM link to be effective on every actual monthly CRSP date. */
proc sql;
  create table return_months as
  select distinct c.obsid,c.gvkey,c.fyear,c.datadate,c.permno,c.permco,
         intnx('month',m.date,0,'b') as mth format=date9.,m.date,m.ret
  from ret_choice as c
  inner join crsp.msf as m on m.permno=c.permno and m.permco=c.permco
  inner join linkbase as l on l.gvkey=c.gvkey and l.lpermno=c.permno
                           and l.lpermco=c.permco
  where intnx('month',m.date,0,'b') between
          intnx('month',c.datadate,-8,'b') and intnx('month',c.datadate,3,'b')
    and (l.linkdt=.B or l.linkdt<=m.date)
    and (l.linkenddt=.E or l.linkenddt>=m.date)
    and m.date between '01jan1962'd and '31dec1991'd;
  create table return_counts as
  select obsid,gvkey,fyear,datadate,count(*) as n_rows,
         count(distinct mth) as n_months,count(ret) as n_returns,
         min(mth) as first_month,max(mth) as last_month
  from return_months
  group by obsid,gvkey,fyear,datadate;
quit;
proc sort data=return_months out=return_months_sort;
  by obsid mth;
run;
data return_products;
  set return_months_sort;
  by obsid;
  retain grossret;
  if first.obsid then grossret=1;
  if not missing(ret) then grossret=grossret*(1+ret);
  if last.obsid then do;
    actual_ret=grossret-1;
    output;
  end;
  keep obsid gvkey fyear datadate actual_ret;
run;
proc sql;
  create table return_valid as
  select p.obsid,p.gvkey,p.fyear,p.datadate,p.actual_ret
  from return_products as p inner join return_counts as c
    on p.obsid=c.obsid
  where c.n_rows=12 and c.n_months=12 and c.n_returns=12
    and c.first_month=intnx('month',p.datadate,-8,'b')
    and c.last_month=intnx('month',p.datadate,3,'b');
quit;

/* Join construction components and retain complete earnings, currency,
   company market equity and actual-return observations. */
proc sort data=funda_usd; by obsid; run;
proc sort data=mve_stats; by obsid; run;
proc sort data=return_valid; by obsid; run;
data analysis_raw;
  merge funda_usd(in=a) mve_stats(in=b) return_valid(in=c);
  by obsid;
  if a and b and c and fx_obs=1 and not missing(ib_usd_th)
     and n_issues>0 and n_valid_issues=n_issues and company_mve_th>0
     and primary_mve_th>0 and not missing(actual_ret);
  permno=primary_permno;
  x=ib_usd_th/company_mve_th;
  r_actual=actual_ret;
  company_to_primary=company_mve_th/primary_mve_th;
  format datadate anchor_mth anchor_date date9.;
  label x='X: item 18 proxy / company MVE at FYE-9'
        r_actual='Actual 12-month CRSP return'
        company_mve_th='Company MVE in USD thousands'
        primary_mve_th='Primary linked issue MVE in USD thousands'
        company_to_primary='Company MVE / primary issue MVE';
run;
proc sort data=analysis_raw out=analysis_key;
  by gvkey fyear datadate;
run;
data analysis_key;
  set analysis_key;
  by gvkey fyear;
  if first.fyear and last.fyear;
  obsid=_n_;
run;

/* Within-year tail exclusion is applied to the full actual-return sample and
   to each source variable before news partitions. */
proc means data=analysis_key nway noprint;
  class fyear;
  var r_actual;
  output out=ret_cut(drop=_type_ _freq_) p1=r_p1 p99=r_p99;
run;
proc means data=analysis_key nway noprint;
  class fyear;
  var x;
  output out=x_cut(drop=_type_ _freq_) p1=x_p1 p99=x_p99;
run;
proc sort data=analysis_key; by fyear; run;
proc sort data=ret_cut; by fyear; run;
proc sort data=x_cut; by fyear; run;
data trim_cut;
  merge ret_cut x_cut;
  by fyear;
run;
proc sort data=trim_cut; by fyear; run;
data trimmed0;
  merge analysis_key(in=a) trim_cut;
  by fyear;
  if a;
  keep_r=(r_actual>=r_p1 and r_actual<=r_p99);
  keep_x=(x>=x_p1 and x<=x_p99);
  keep_both=(keep_r and keep_x);
run;
data trimmed;
  set trimmed0;
  if keep_both;
run;

/* Supporting generator: one pooled OLS of actual R on X over the full trimmed
   actual sample. The observed firm-year residual is retained only in WORK. */
proc reg data=trimmed outest=gen_est;
  model r_actual=x;
  output out=gen_pred p=fitted r=eps;
run;
quit;
data _null_;
  set gen_est;
  if _TYPE_='PARMS' then do;
    call symputx('gen_b0',Intercept);
    call symputx('gen_b1',x);
  end;
run;

/* Seeded within-year cyclic rotation of a random ordering is a one-to-one
   residual permutation and a derangement whenever the year has >=2 firms. */
data gen_shuffle;
  set gen_pred;
  if _n_=1 then call streaminit(20261103);
  shuffle_u=rand('uniform');
run;
proc sort data=gen_shuffle out=gen_order;
  by fyear shuffle_u gvkey;
run;
data sim_assignment0;
  set gen_order;
  by fyear;
  retain first_obs prev_eps prev_gvkey n_year;
  length sim_source_gvkey $16;
  if first.fyear then do;
    first_obs=obsid;
    n_year=0;
  end;
  n_year+1;
  if not first.fyear then do;
    target_obs=obsid;
    sim_eps=prev_eps;
    sim_source_gvkey=prev_gvkey;
    output;
  end;
  prev_eps=eps;
  prev_gvkey=gvkey;
  if last.fyear and n_year>=2 then do;
    target_obs=first_obs;
    sim_eps=eps;
    sim_source_gvkey=gvkey;
    output;
  end;
  keep target_obs fyear sim_eps sim_source_gvkey;
run;
proc sql;
  create table sim_assignment as
  select a.target_obs,a.fyear,a.sim_eps,a.sim_source_gvkey,
         g.gvkey as target_gvkey
  from sim_assignment0 as a inner join gen_pred as g
    on a.target_obs=g.obsid;
  create table permutation_audit as
  select fyear,count(*) as n_assignments,
         count(distinct target_obs) as n_unique_targets,
         count(distinct sim_source_gvkey) as n_unique_sources,
         sum(case when target_gvkey=sim_source_gvkey then 1 else 0 end) as n_self
  from sim_assignment group by fyear;
  create table simulation as
  select g.*,a.sim_eps,a.sim_source_gvkey,
         &gen_b0 + (&gen_b1*g.x) + a.sim_eps as r_sim
  from gen_pred as g left join sim_assignment as a
    on g.obsid=a.target_obs;
quit;

/* Construction and sample audit totals; published counts are comparison
   results and are never used as selection rules. */
proc sql;
  title 'Provider-period currency conversion counts by own-period CURCD';
  select curcd,count(*) as annual_rows,
         sum(fx_obs=1 and not missing(ib_usd_th)) as converted_rows,
         sum(fx_obs=0) as no_period_rate_rows
  from funda_usd group by curcd order by curcd;
  title 'Construction attrition counts';
  select 'funda_annual' as stage length=24,count(*) as n
    from funda_annual
  union all select 'funda_usd_complete',count(*) from funda_usd
    where fx_obs=1 and not missing(ib_usd_th)
  union all select 'anchor_candidates',count(distinct obsid) from anchor_candidates
  union all select 'unique_return_link',count(distinct obsid) from ret_choice
  union all select 'company_mve_complete',count(*) from mve_stats
    where n_issues>0 and n_valid_issues=n_issues and company_mve_th>0
  union all select 'complete_actual_window',count(*) from return_valid
  union all select 'actual_complete_join',count(*) from analysis_raw
  union all select 'after_within_year_trim',count(*) from trimmed
  ;
  title 'Source sample counts by fiscal year';
  select fyear,count(*) as n
  from analysis_raw group by fyear order by fyear;
  title 'Ambiguous top-ranked anchor links';
  select count(*) as ambiguous_firm_years
  from anchor_rank_count where n_top_permno ne 1 or n_top_permco ne 1;
  title 'Trim counts and retained rows by fiscal year';
  select fyear,count(*) as before_n,sum(keep_r=0) as excluded_return,
         sum(keep_x=0) as excluded_x,sum(keep_both=0) as excluded_union,
         sum(keep_both=1) as retained
  from trimmed0 group by fyear order by fyear;
  title 'Actual and simulated sign partitions';
  select fyear,
         sum(r_actual<0) as actual_bad,
         sum(r_actual>=0) as actual_good,
         sum(r_sim<0) as simulated_bad,
         sum(r_sim>=0) as simulated_good,
         sum(missing(r_sim)) as missing_sim
  from simulation group by fyear order by fyear;
  title 'Residual assignment audit by year';
  select * from permutation_audit order by fyear;
  title 'Company versus primary issue market equity audit';
  select count(*) as n,
         sum(abs(company_mve_th-primary_mve_th)>
             1e-8*max(abs(company_mve_th),abs(primary_mve_th))) as n_different,
         mean(company_to_primary) as mean_company_to_primary,
         min(company_to_primary) as min_company_to_primary,
         max(company_to_primary) as max_company_to_primary
  from analysis_raw;
quit;
data _null_;
  set permutation_audit;
  if n_assignments ne n_unique_targets or
     n_assignments ne n_unique_sources or n_self ne 0 then do;
    put 'ERROR: residual assignment is not a one-to-one within-year derangement';
    abort cancel;
  end;
run;
title 'Final analysis-input distributions';
proc means data=simulation n mean std var median min max;
  var r_actual r_sim x company_mve_th ib_usd_th company_to_primary;
run;
title;

/* Export one unique firm-year row with both partition variables. */
data out.basu_replication_1963_1990(label='Basu 1963-1990 actual and simulated returns');
  set simulation;
  keep gvkey fyear datadate permno permco curcd ib ib_usd_th fx_obs
       fx_report_month fx_gbp_to_reporting fx_gbp_to_usd
       fx_quote_reporting fx_quote_usd
       anchor_mth anchor_date ret_first ret_last primary_mve_th company_mve_th
       company_to_primary x r_actual r_sim sim_source_gvkey;
  label gvkey='Compustat global company key'
        fyear='Compustat fiscal year'
        datadate='Actual fiscal period end'
        permno='Date-valid CCM primary return security'
        permco='Date-valid CRSP company identifier'
        curcd='Own-period Compustat reporting currency'
        ib='Compustat income before extraordinary items (item 18 proxy)'
        ib_usd_th='Item 18 proxy in USD thousands'
        fx_obs='USD identity or provider EXRAT12M rate available'
        fx_report_month='Statement reporting month for EXRAT12M'
        fx_gbp_to_reporting='GBP to reporting-currency EXRAT12M leg'
        fx_gbp_to_usd='GBP to USD EXRAT12M leg'
        fx_quote_reporting='Rows in reporting-currency quote group'
        fx_quote_usd='Rows in USD quote group'
        anchor_mth='FYE-9 anchor month'
        anchor_date='Actual CRSP anchor quote date'
        ret_first='First required monthly return period'
        ret_last='Last required monthly return period'
        primary_mve_th='Primary linked issue MVE in USD thousands'
        company_mve_th='Company MVE in USD thousands'
        company_to_primary='Company MVE / primary issue MVE'
        x='Scaled earnings X'
        r_actual='Actual CRSP 12-month return'
        r_sim='One-draw simulated return'
        sim_source_gvkey='Same-year source firm for simulated residual';
  format datadate fx_report_month anchor_mth anchor_date ret_first ret_last date9.;
run;
